# Demo 1

<table><tr>
    <td><img src="image/esi-sba.png" width="100" height="100"></td>
    <td><img src="image/mongo.png" width="100" height="100"></td>
    </tr></table>

### Lancer un conteneur de MONGO
   
! docker-compose up –d


#### installer pymongo afin de se connecter au cluster mongodb

! pip install pymongo

In [4]:
from pymongo import MongoClient

### We are then free to add useful methods as needed. These methods are wrappers for core pymongo collection methods including the following:
* find()
* find_one()
* insert_one()
* insert_many()
* update_one()
* update_many()
* delete_one()
* delete_many()

### Créer un Mongo Client afin d'exécuter nos requêtes sur MONGODB Server

In [5]:
client = MongoClient('mongodb://localhost:27017/')

###  Afficher les différentes  databases

In [6]:
client.list_database_names() 

ServerSelectionTimeoutError: localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 692c4563ce9493b3fdf6875e, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

### Créer une "database variable" pour travailler avec la BDD "db1"

In [ ]:
db = client["db1"] #use db1

### Récupérer les collections de la BDD "db1"

In [ ]:
db.list_collection_names()

###  Insérer un document dans la collection "users"

In [ ]:
db["users"].drop()
db["comments"].drop()

db["users"].insert_one({
                    "_id":1,
                  "name": "Ali", 
                  "login": "ali89", 
                  "age": 29,
                  "address": {
                               "street": "12 gambitta", 
                               "city": "sba"
                             },
                  "job" : ['web', 'BDD']
                    })

###  #Docs

In [ ]:
db["users"].count_documents({}) # db.user.countDocuemnts()

In [ ]:
db["users"].find_one()

### insertion multiple

In [ ]:
userlist = [
  { "_id":2,"name":"Ahmed","login": "Ahmed77f", "age": 28,"address":{"street":"10 maconi","city": "sba"},"job" : ["web", "PHP"]},
  { "_id":3,"name":"said","login": "saidyy", "age": 25,"address":{"city": "oran"},"job" : ["web", "JAVA","PHP"]}, 
  { "_id":4,"name":"amine","login": "amine77","age": 18},
  { "_id":5,"name":"said","login": "ss.ki", "age": 17,"address":{"city": "sba"},"job" : ["python", "JAVA","PHP"]}, 
  
]

db["users"].insert_many(userlist)

db["users"].count_documents({})

In [ ]:
db["users"].find()

### Afficher tous les docs

In [ ]:
# Display all records
for user in db["users"].find():
  print(user,'\n') 

### Update

In [ ]:
db["users"].update_one( 
                        {'name': 'Ali'},
                        { "$inc" : { "age" : 1 } } 
                   )


### Update multiple

In [ ]:
db["users"].update_many( 
                        {'age': {'$lte':18}},
                        { '$set' : { 'job' : [] } } 
                   )

### .find( filtre , projection )

In [ ]:
for user in db["users"].find({}, {"name": 1,"age": 1}):
  print(user,'\n')


### Recherche exacte

In [ ]:
for user in db["users"].find( {'login' : "ali89"} , {'login':1,'name' : 1, 'age' : 1}):
  print(user,'\n')

In [ ]:
for user in db["users"].find( {'login' : {'$regex': 'Ali89', '$options':'i'}} , {'login':1,'name' : 1, 'age' : 1}):
  print(user,'\n')

### Filtrage dans une imbrication 

In [ ]:
for user in db["users"].find( { 'address.city' : 'sba'}):
  print(user,'\n')

### Filtrage avec opération 
* \\$gt, \\$gte, \\$lt, \\$lte, \\$ne, \\$in, \\$nin, \\$or, \\$and, \\$regex, \\$exists, \\$type, \\$size, …

In [ ]:
for user in db["users"].find( { 'age' :{ '$gt' : 28 }},  {"_id": 0,"name": 1,"age": 1} ):
  print(user,'\n')

In [ ]:
for user in db["users"].find( { 'age' :{ '$in' : [18,20,30] }},  {"_id": 0,"name": 1,"age": 1} ):
  print(user,'\n')

In [ ]:
for user in db["users"].find( { '$or': [ {'age' : { '$gt' : 20 } }, { 'address.city' : 'sba'} ] } ):
     print(user,'\n')

In [ ]:
for user in db["users"].find( { 'name' : { '$exists': 'false'} } ):
    print(user,'\n')

In [ ]:
for user in db["users"].find( { 'login' :{ '$regex' : '^a.m', '$options':'i'}},  {"_id": 0,"name": 1,"login": 1} ):
  print(user,'\n')

In [ ]:
db["users"].create_index( [('name', "text"), ('login', "text")] )

for user in db["users"].find( {'$text':{ '$search': 'ali ahmed ss.ki'} } ):
       print(user,'\n')

### Filtrage sur un tableau

In [ ]:
for user in db["users"].find( { 'job' : 'BDD'} ): #dans la liste
  print(user,'\n')

In [ ]:
for user in db["users"].find( {'job.1' : 'BDD'} ): #2éme place de la liste
  print(user,'\n')

In [ ]:
for user in db["users"].find( { 'job' : ['web', 'BDD']} ):  #recherche exacte
  print(user,'\n')

### SORT,  DISTINCT, COUNT

In [ ]:
for user in db["users"].find().sort("name"):
    print(user,'\n')

In [ ]:
for user in db["users"].distinct( 'age'):
    print(user,'\n')

In [ ]:
print('# docs in Users collection =',db["users"].count_documents({}),'\n')

print('# user whose age is 18=',db["users"].count_documents({ 'age' : 30 }))
   

### Pipeline: aggregate

In [ ]:
result=db["users"].aggregate([  
                              {'$match' : {"address.city" : 'sba'}},
                              {'$project' : {"login" : 1, "age" : 1}}, 
                              {'$sort' : {"age" : 1, "login" : -1}}
                             ]);

for user in result:
    print(user,'\n')

### Pipeline: \\$Unwind

In [ ]:
result=db["users"].aggregate([  
                              {'$unwind' : "$job"},
                             {'$project' : {'_id':0, "login" : 1, "age" : 1, "job":1}},
                            ]);

for user in result:
    print(user,'\n')

### Pipeline: \\$GROUP

In [ ]:
result=db["users"].aggregate([  
                              {'$group':{"_id" : "$age", "res": {'$sum' : 1}}}

                            ]);

for user in result:
    print(user,'\n')

In [ ]:
result=db["users"].aggregate([  
                              {'$match': {"address.city" : "sba"}},
                              {'$unwind' : "$job"},
                              {'$group' : {"_id" : "$job", "moy": {'$avg': "$age"}} },
                              {'$match' : {"moy" : {'$gt' : 10}} },
                              {'$sort' : { "moy" : -1} } 

                            ]);

for user in result:
    print(user,'\n')

### Lookup (join between collection)

In [ ]:
commentslist = [
  {"_id":1, "content":"sjkjzk zkzkz kzkz","userid": 1},
  {"_id":2, "content":"szid zoeoez","userid": 1},
  { "_id":3,"content":"uu ueue ","userid": 1}, 
  { "_id":4,"content":"skks kzkz kzkz","userid": 2},
  { "_id":5,"content":"dkkd kdkd","userid": 2},
  { "_id":6,"content":"js skks ksk ","userid": 2}, 
   {"_id":7, "content":"ksk sids ","userid": 3}, 
]

db["comments"].insert_many(commentslist)

for comment in db["comments"].find():
    print(comment,'\n')

In [ ]:
result2=db["users"].aggregate([
           {
              '$lookup': {
              'from': "comments",
              'localField': "_id",
              'foreignField': "userid",
               'as': "commentdetails",
             }
     }
])


for user in result2:
    print(user,'\n')